# CN7 대회용 라벨 데이터에 Timestamp 연결

대회용 `moldset_labeled_cn7.csv`의 `Unnamed: 0`을 원본 `moldset_labeled.csv`의 같은 행 번호와 연결합니다. 대회용 센서값(표준화된 값)은 변경하지 않고, 원본의 시각·제품명·RH/LH·불량 사유만 추가합니다.

출력은 RH/LH 각각 한 행입니다. 같은 Timestamp의 두 행을 독립적인 Shot으로 세지 않도록 주의하세요. Timestamp로 추정한 Run은 실제 설비 가동 기록과 다를 수 있습니다. 원본에만 있는 CN7 `초기허용불량`은 이 대회용 부분집합에 포함되지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent

competition = pd.read_csv(root / 'data/moldset_labeled_cn7.csv')
original = pd.read_csv(root / 'data/original/moldset_labeled.csv')

key = 'Unnamed: 0'
assert competition[key].is_unique and original[key].is_unique
assert competition[key].notna().all()
assert competition[key].isin(original[key]).all()
matched = original.set_index(key).loc[competition[key]].reset_index()
assert matched['PART_NAME'].str.startswith('CN7').all()
assert matched['PassOrFail'].equals(competition['PassOrFail'])
assert matched['TimeStamp'].notna().all()

metadata = matched[['TimeStamp', 'PART_NAME', 'PART_NO', 'EQUIP_CD', 'Reason']].copy()
metadata['cavity_side'] = metadata['PART_NAME'].str.extract(r'(LH|RH)\s*$')
metadata['Reason'] = metadata['Reason'].fillna('정상')
assert metadata['cavity_side'].notna().all()
result = pd.concat([competition[[key]].rename(columns={key: 'source_row_id'}),
                    metadata.reset_index(drop=True),
                    competition.drop(columns=[key, 'PassOrFail']).reset_index(drop=True),
                    competition[['PassOrFail']].reset_index(drop=True)], axis=1)
result['TimeStamp'] = pd.to_datetime(result['TimeStamp'])
result = result.sort_values(['TimeStamp', 'source_row_id']).reset_index(drop=True)

output = root / 'data/moldset_labeled_cn7_with_timestamp.csv'
result.to_csv(output, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
print(f'Saved: {output}')
print(f'Rows: {len(result):,}; unique timestamps/shots: {result.TimeStamp.nunique():,}')
print(result.groupby(['Reason', 'PassOrFail']).size())
result[['TimeStamp', 'PART_NAME', 'cavity_side', 'Reason', 'PassOrFail']].head(10)